# 06 — Statistical Significance & Wavelet Filter Analysis

Três análises para suporte ao artigo:

1. **Wilcoxon signed-rank** por arquitetura — testa se cada modo wavelet supera o baseline `raw` nos 25 ativos  
2. **Correção BH (Benjamini-Hochberg)** para múltiplas comparações  
3. **Análise dos filtros aprendidos** — extrai e compara os filtros low-pass do `learned_wavelet` com o db4 de referência

In [ ]:
import sys, warnings, itertools
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

BASE = Path(".").resolve()
sys.path.insert(0, str(BASE))
sys.path.insert(0, str(BASE.parent.parent))

from src.evaluation import ResultsManager, WilcoxonComparison

RESULTS_DIR      = Path("results")
SAVED_MODELS_DIR = Path("saved_models")
FEATURE_MODE = "features"   # features | ohlcv
ARCHS      = ["CNN", "LSTM", "CNN_LSTM", "Transformer"]
MODES      = ["raw", "db4", "learned_wavelet"]
MODE_PAIRS = [("db4", "raw"), ("learned_wavelet", "raw"), ("learned_wavelet", "db4")]

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

In [ ]:
rm    = ResultsManager(RESULTS_DIR)
_df   = rm.load_all_results()
df    = _df[_df["feature_mode"] == FEATURE_MODE].copy().reset_index(drop=True)
dl_df = df[df["mode"].isin(MODES)].copy()

print(f"feature_mode filter: {FEATURE_MODE!r}  ({len(df)}/{len(_df)} experiments)")
print(f"Experimentos DL carregados : {len(dl_df)}")
print(f"Ativos                     : {dl_df['ticker'].nunique()}")
print(f"Arquiteturas               : {dl_df['model_name'].nunique()}")
print(f"\nMétricas OOS disponíveis:")
oos_cols = [c for c in dl_df.columns if "oos" in c]
print(oos_cols)
dl_df[["ticker","model_name","mode","fin_oos_f1_macro","fin_oos_sharpe"]].head(6)


## 1. Significância Estatística — Wilcoxon + Correção BH

**Design**: para cada par `(modo_A vs modo_B)` dentro de cada arquitetura, o teste de Wilcoxon signed-rank usa os 25 ativos como observações pareadas. Isso controla o efeito de ativo — o mesmo ativo é avaliado nos dois modos.

**Total de testes**: 4 arquiteturas × 3 pares = **12 testes** → correção BH com FDR = 0.05.

In [ ]:
def run_wilcoxon_bh(results_df, metric, mode_pairs, archs, group_col="ticker"):
    """Wilcoxon por arquitetura × par de modos com correção BH."""
    records = []
    for arch in archs:
        arch_df = results_df[results_df["model_name"] == arch]
        for mode_a, mode_b in mode_pairs:
            res = WilcoxonComparison.compare_modes(
                arch_df, metric=metric,
                baseline_mode=mode_b,
                comparison_modes=[mode_a],
                group_col=group_col,
            )
            if not res.empty:
                res["architecture"] = arch
                records.append(res)

    tests = pd.concat(records, ignore_index=True)

    # BH correction nos p-valores válidos
    valid = tests["p_value"].notna()
    p_bh  = np.ones(len(tests))
    rej   = np.zeros(len(tests), dtype=bool)
    if valid.sum() >= 2:
        rej[valid], p_bh[valid], _, _ = multipletests(
            tests.loc[valid, "p_value"], method="fdr_bh"
        )
    tests["p_bh"]          = p_bh
    tests["significant_bh"] = rej
    return tests


# ── OOS F1-macro ──────────────────────────────────────────────────────────
tests_f1 = run_wilcoxon_bh(dl_df, "fin_oos_f1_macro", MODE_PAIRS, ARCHS)

SHOW = ["architecture","comparison","n_pairs","median_diff","mean_diff","effect_size","p_value","p_bh","significant_bh"]
print("=== OOS F1-macro — Wilcoxon signed-rank (BH-corrected, α=0.05) ===")
print(tests_f1[SHOW].to_string(index=False))

In [ ]:
# ── OOS Sharpe ────────────────────────────────────────────────────────────
tests_sharpe = run_wilcoxon_bh(dl_df, "fin_oos_sharpe", MODE_PAIRS, ARCHS)

print("=== OOS Sharpe — Wilcoxon signed-rank (BH-corrected, α=0.05) ===")
print(tests_sharpe[SHOW].to_string(index=False))


In [ ]:
# ── Heatmap effect size — F1-macro ────────────────────────────────────────
def plot_effect_heatmap(tests, metric_label, ax):
    pivot_e = tests.pivot_table(index="architecture", columns="comparison", values="effect_size")
    pivot_s = tests.pivot_table(index="architecture", columns="comparison", values="significant_bh")

    sns.heatmap(
        pivot_e, annot=True, fmt=".2f", cmap="RdYlGn",
        center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
        annot_kws={"size": 10},
    )
    # Marca * nas células significativas após BH
    for i, arch in enumerate(pivot_e.index):
        for j, cmp in enumerate(pivot_e.columns):
            try:
                if pivot_s.loc[arch, cmp]:
                    ax.text(j + 0.85, i + 0.2, "*", fontsize=14, color="black", ha="center")
            except KeyError:
                pass
    ax.set_title(f"Effect size (rank-biserial) — {metric_label}\n* = significativo após BH (α=0.05)")
    ax.set_xlabel("")
    ax.set_ylabel("")


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_effect_heatmap(tests_f1,     "OOS F1-macro",  axes[0])
plot_effect_heatmap(tests_sharpe, "OOS Sharpe",     axes[1])
plt.tight_layout()
plt.show()

## 2. Análise dos Filtros Aprendidos

Extrai o filtro low-pass de cada modelo `learned_wavelet` salvo (fold 4 = último fold IS) e compara com o db4 de referência. Avalia:
- **Similaridade coseno** com db4  
- **Resposta em frequência** (FFT)  
- **Consistência entre ativos** (std dos filtros por arquitetura)

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
import pywt

from models.LWT.learned_wavelet_dwt_qmf  import LearnedWaveletDWT1D_QMF
from models.LWT.learned_wavelet_pair_qmf import LearnedWaveletPair1D_QMF
from models.dl_utils import SinusoidalPositionalEncoding, TransformerBlock

CUSTOM_OBJECTS = {
    "LearnedWaveletDWT1D_QMF":      LearnedWaveletDWT1D_QMF,
    "LearnedWaveletPair1D_QMF":     LearnedWaveletPair1D_QMF,
    "SinusoidalPositionalEncoding": SinusoidalPositionalEncoding,
    "TransformerBlock":             TransformerBlock,
}


def get_wavelet_filters(model):
    """Retorna {level: h_array} para cada nível do frontend wavelet."""
    wl = next((l for l in model.layers if isinstance(l, LearnedWaveletDWT1D_QMF)), None)
    if wl is None:
        return None
    result = {}
    for lvl, pair in enumerate(wl.pairs):
        t     = pair._make_t()
        scale = tf.nn.softplus(pair.raw_scale) + 1e-3
        t_adj = (t - pair.translation) / scale
        z     = pair.base_net(t_adj)
        h     = pair.low_head(z)
        h     = pair._normalize_h(h)
        result[lvl] = h.numpy().flatten()
    return result


_models_root = SAVED_MODELS_DIR / FEATURE_MODE
TICKERS = sorted([d.name for d in _models_root.iterdir() if d.is_dir()]) if _models_root.exists() else []
filters  = {}   # (ticker, arch) -> {level: h_array}
missing  = []

for ticker in TICKERS:
    for arch in ARCHS:
        model_path = _models_root / ticker / f"{arch}_learned_wavelet" / "fold_4.keras"
        if not model_path.exists():
            missing.append((ticker, arch))
            continue
        try:
            model = tf.keras.models.load_model(
                str(model_path), custom_objects=CUSTOM_OBJECTS
            )
            f = get_wavelet_filters(model)
            if f:
                filters[(ticker, arch)] = f
            tf.keras.backend.clear_session()
        except Exception as e:
            missing.append((ticker, arch))
            print(f"  ERRO {ticker}/{arch}: {e}")

print(f"Filtros extraídos : {len(filters)} / {len(TICKERS) * len(ARCHS)}")
if missing:
    print(f"Missing           : {len(missing)} modelos")


In [ ]:
# ── Similaridade coseno com db4 ───────────────────────────────────────────
h_db4_ref = np.array(pywt.Wavelet("db4").dec_lo, dtype=np.float32)

def cosine_sim(a, b):
    min_len = min(len(a), len(b))
    a_n = a[:min_len] / (np.linalg.norm(a[:min_len]) + 1e-8)
    b_n = b[:min_len] / (np.linalg.norm(b[:min_len]) + 1e-8)
    return float(np.dot(a_n, b_n))


sim_records = [
    {"ticker": t, "architecture": a,
     "level": lvl, "cos_sim_db4": cosine_sim(h, h_db4_ref),
     "kernel_size": len(h)}
    for (t, a), fdict in filters.items()
    for lvl, h in fdict.items()
]
sim_df = pd.DataFrame(sim_records)

print("Similaridade coseno com db4 — Level 0 (low-pass, nível 1):")
lvl0 = sim_df[sim_df["level"] == 0]
print(lvl0.groupby("architecture")["cos_sim_db4"].agg(["mean","std","min","max"]).round(3).to_string())

if sim_df["level"].max() > 0:
    print("\nSimilaridade coseno com db4 — Level 1 (low-pass, nível 2):")
    lvl1 = sim_df[sim_df["level"] == 1]
    print(lvl1.groupby("architecture")["cos_sim_db4"].agg(["mean","std","min","max"]).round(3).to_string())

In [ ]:
# ── Filtros no domínio temporal — todos os ativos × arquitetura ───────────
fig, axes = plt.subplots(1, len(ARCHS), figsize=(16, 4), sharey=False)

for ax, arch in zip(axes, ARCHS):
    arch_items = [(t, v[0]) for (t, a), v in filters.items() if a == arch]
    for _, h in arch_items:
        k = len(h)
        ax.plot(np.linspace(0, 1, k), h, color="#3498db", alpha=0.25, linewidth=0.9)

    # Média dos filtros aprendidos
    if arch_items:
        # Agrupa por kernel_size (pode variar entre runs)
        sizes = {len(h) for _, h in arch_items}
        for ks in sizes:
            hs = np.array([h for _, h in arch_items if len(h) == ks])
            mean_h = hs.mean(axis=0)
            ax.plot(np.linspace(0, 1, ks), mean_h,
                    color="#2c3e50", linewidth=2, label="média LWT")

    ax.plot(np.linspace(0, 1, len(h_db4_ref)), h_db4_ref,
            color="#e74c3c", linewidth=2, linestyle="--", label="db4")
    ax.set_title(arch, fontsize=11)
    ax.set_xlabel("Coeficiente normalizado")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Amplitude")
fig.suptitle("Filtros Low-pass Aprendidos (nível 1) vs db4 — 25 ativos", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Resposta em frequência ────────────────────────────────────────────────
N_FFT = 256
freqs   = np.fft.rfftfreq(N_FFT, d=1.0)
H_db4   = np.abs(np.fft.rfft(h_db4_ref, n=N_FFT))

fig, axes = plt.subplots(1, len(ARCHS), figsize=(16, 4), sharey=False)

for ax, arch in zip(axes, ARCHS):
    arch_items = [(t, v[0]) for (t, a), v in filters.items() if a == arch]
    H_list = []
    for _, h in arch_items:
        H = np.abs(np.fft.rfft(h, n=N_FFT))
        ax.plot(freqs, H, color="#3498db", alpha=0.2, linewidth=0.8)
        H_list.append(H)

    if H_list:
        H_mean = np.mean(H_list, axis=0)
        ax.plot(freqs, H_mean, color="#2c3e50", linewidth=2, label="média LWT")

    ax.plot(freqs, H_db4, color="#e74c3c", linewidth=2, linestyle="--", label="db4")
    ax.set_title(arch, fontsize=11)
    ax.set_xlabel("Frequência normalizada")
    ax.set_xlim(0, 0.5)
    ax.legend(fontsize=8)

axes[0].set_ylabel("|H(f)|")
fig.suptitle("Resposta em Frequência — Filtro Low-pass Nível 1", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Consistência entre ativos (std dos filtros por arquitetura) ───────────
fig, axes = plt.subplots(1, len(ARCHS), figsize=(16, 4), sharey=True)

for ax, arch in zip(axes, ARCHS):
    by_size = {}
    for (t, a), fdict in filters.items():
        if a != arch:
            continue
        h = fdict[0]
        by_size.setdefault(len(h), []).append(h)

    if not by_size:
        continue

    # Usa o grupo de kernel_size mais frequente
    ks     = max(by_size, key=lambda k: len(by_size[k]))
    hs     = np.array(by_size[ks])
    mean_h = hs.mean(axis=0)
    std_h  = hs.std(axis=0)
    x      = np.linspace(0, 1, ks)

    ax.plot(x, mean_h, color="#2c3e50", linewidth=2, label="média")
    ax.fill_between(x, mean_h - std_h, mean_h + std_h,
                    alpha=0.3, color="#3498db", label="±1σ")
    ax.plot(np.linspace(0, 1, len(h_db4_ref)), h_db4_ref,
            color="#e74c3c", linewidth=1.5, linestyle="--", label="db4")

    ax.set_title(f"{arch}\nstd médio = {std_h.mean():.4f}  (n={len(hs)} ativos)", fontsize=10)
    ax.set_xlabel("Coeficiente")
    ax.legend(fontsize=7)

axes[0].set_ylabel("Amplitude")
fig.suptitle(
    "Consistência entre Ativos — Filtro Low-pass Nível 1\n"
    "std baixo = padrão universal; std alto = filtro idiossincráico por ativo",
    fontsize=11,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabela resumo final ───────────────────────────────────────────────────
print("\n=== RESUMO PARA O ARTIGO ===")

print("\n[1] Wilcoxon + BH — OOS F1-macro")
sig_f1 = tests_f1[tests_f1["significant_bh"]][["architecture","comparison","median_diff","effect_size","p_bh"]]
print(sig_f1.to_string(index=False) if not sig_f1.empty else "  Nenhuma comparação significativa após BH")

print("\n[2] Wilcoxon + BH — OOS Sharpe")
sig_s = tests_sharpe[tests_sharpe["significant_bh"]][["architecture","comparison","median_diff","effect_size","p_bh"]]
print(sig_s.to_string(index=False) if not sig_s.empty else "  Nenhuma comparação significativa após BH")

print("\n[3] Similaridade coseno média (LWT level-0 vs db4):")
print(lvl0.groupby("architecture")["cos_sim_db4"].mean().round(3).to_string())